<a href="https://colab.research.google.com/github/athfizh/PemMes26_05_Hafizh/blob/main/JS03/JS03-TugasLab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---


# **Deskripsi Tugas**


---

Pada tugas pratikum ini Anda akan menggunakan data "Wisconsin Breast Cancer". Data tersebut terdiri dari 569 data yang digunakan untuk mendiagnonis jenis kanker Malignant (M) dan Benign (B). Tugas Anda adalah,

1. Pisahkan antara variabel yang dapat digunakan dan variabel yang tidak dapat digunakan.

2. Lakukan proses encoding pada kolom "diagnosis".

3. Lakukan proses standardisasi pada semua kolom yang memiliki nilai numerik.

4. Lakukan proses seleksi fitur. Anda dapat menggunakan SelectKBest.

5. Lakukan proses pengujian dengan model Logistic Regression seperti pada praktikum 1.

6. Anda dapat menggunakan model pipeline untuk mempermudah perkejaan Anda.

7. Berdasarkan hasil analisa Anda, berapa jumlah fitur terbaik yang dapat digunakan? Apa saja fitur tersebut?

---


# **Langkah 0 - Persiapan Lingkungan & Load Data**


---

In [6]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# Load dataset
df = pd.read_csv('wbc.csv')
print("Ukuran awal dataset:", df.shape)

Ukuran awal dataset: (569, 33)


---


# **Langkah 1 - Memisahkan Variabel (Drop Kolom)**


---

In [7]:
# Menghapus variabel yang tidak dapat digunakan
df_cleaned = df.drop(columns=['id', 'Unnamed: 32'])
print("Ukuran dataset setelah drop kolom:", df_cleaned.shape)

Ukuran dataset setelah drop kolom: (569, 31)


---


# **Langkah 2 - Encoding Kolom "diagnosis"**


---

In [8]:
le = LabelEncoder()
df_cleaned['diagnosis'] = le.fit_transform(df_cleaned['diagnosis'])

# Memisahkan fitur (X) dan target (y)
X = df_cleaned.drop(columns=['diagnosis'])
y = df_cleaned['diagnosis']

# Split data menjadi training dan testing (80:20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

---


# **Langkah 3, 4, 5, dan 6 - Membangun Pipeline**


---

In [9]:
# Membuat pipeline yang berisi standardisasi, seleksi fitur, dan model
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(score_func=f_classif)),
    ('classifier', LogisticRegression())
])

---


# **Langkah 7 - Mencari Jumlah Fitur Terbaik & Evaluasi**


---

In [10]:
# Mencari nilai k terbaik menggunakan GridSearchCV
param_grid = {
    'selector__k': range(1, len(X.columns) + 1)
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

# Mendapatkan k terbaik
best_k = grid_search.best_params_['selector__k']
best_score = grid_search.best_score_
print(f"Jumlah fitur terbaik (k) yang direkomendasikan: {best_k}")
print(f"Akurasi Validasi Silang (Cross-validation) terbaik: {best_score:.4f}")

# Mengevaluasi model terbaik pada data testing
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Akurasi pada Data Testing: {test_accuracy:.4f}")

# Menampilkan nama-nama fitur terbaik yang terpilih
selector = best_model.named_steps['selector']
fitur_terpilih = X.columns[selector.get_support()].tolist()

print("\n--- Fitur Terbaik yang Digunakan ---")
for i, fitur in enumerate(fitur_terpilih, 1):
    print(f"{i}. {fitur}")

Jumlah fitur terbaik (k) yang direkomendasikan: 17
Akurasi Validasi Silang (Cross-validation) terbaik: 0.9758
Akurasi pada Data Testing: 0.9737

--- Fitur Terbaik yang Digunakan ---
1. radius_mean
2. perimeter_mean
3. area_mean
4. compactness_mean
5. concavity_mean
6. concave points_mean
7. radius_se
8. perimeter_se
9. area_se
10. radius_worst
11. texture_worst
12. perimeter_worst
13. area_worst
14. compactness_worst
15. concavity_worst
16. concave points_worst
17. symmetry_worst


---


# **Hasil Analisis dan Seleksi Fitur**


---

> Dari hasil eksekusi program menggunakan GridSearchCV dan SelectKBest (metode f_classif), didapatkan bahwa jumlah fitur terbaik yang paling optimal untuk digunakan pada model Regresi Logistik adalah 17 fitur. Berdasarkan fitur yang terpilih, terlihat bahwa model cenderung mempertahankan fitur yang mengukur nilai rata rata (mean) dan nilai terburuk (worst) dari hasil pengamatan sel kanker. Hal ini menunjukkan bahwa dimensi ekstrem dari inti sel, seperti luasan (area), keliling (perimeter), dan radius, sangat berpengaruh signifikan secara statistik dalam membedakan tumor ganas dan jinak.
Proses seleksi fitur ini juga berhasil membuang 13 fitur lain yang korelasinya lebih rendah. Pengurangan dimensi ini cukup penting dilakukan untuk menghilangkan noise pada data sehingga beban komputasi menjadi lebih efisien dan model terhindar dari masalah overfitting tanpa harus mengorbankan nilai akurasinya.